
To install, ´conda env create -f environment-HGQ.yml´, or to update env ´conda env update -f environment-HGQ.yml´. Remember to restart kernel.

In [20]:
import os
model_to_test = 'hgq2'
model_revision = 1
hls4ml_revision = 'Vitis_latency_reusefactor1'

base_dir = os.path.abspath(model_to_test)
model_dir = os.path.join(base_dir, str(model_revision))
os.makedirs(model_dir, exist_ok=True)

description = """
# Model Configuration

Testing HGQ2 with a base model from Sergei.
HLS4ML-config: 'strategy = latency' og 'reusefactor = 1' (baseline)


- **Model architecture description**: {model_to_test}
- **Model Revision**: {model_revision}
- **HLS4ML Revision**: {hls4ml_revision}
- **Target Device**: KV260 (xck26-sfvc784-2LV-c)
- **Dataset**: HLS4ML LHC Jets
- **Vivado/Vitis**: 2025.2
"""
output_dir = os.path.join(model_dir, f"hls4ml_prj_{hls4ml_revision}")
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
    f.write(description)

In [21]:
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
import os
from sklearn.metrics import accuracy_score

%matplotlib inline
seed = 0
np.random.seed(seed)

tf.random.set_seed(seed)



In [22]:
os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

In [23]:
# Use absolute paths for data files
x_train_val_path = os.path.join(base_dir, "x_train_val.npy")
x_test_path = os.path.join(base_dir, "x_test.npy")
y_train_val_path = os.path.join(base_dir, "y_train_val.npy")
y_test_path = os.path.join(base_dir, "y_test.npy")
classes_path = os.path.join(base_dir, "classes.npy")

x_train_val = np.load(x_train_val_path)
x_test = np.load(x_test_path)
y_train_val = np.load(y_train_val_path)
y_test = np.load(y_test_path)



In [24]:
# Convert dataset arrays to float32
x_train_val = x_train_val.astype(np.float32)
x_test = x_test.astype(np.float32)
y_train_val = y_train_val.astype(np.float32)
y_test = y_test.astype(np.float32)

Load existing model, or create and train a new

In [25]:
keras_model_path = os.path.join(model_dir, f"model_HGQ.keras")

import hgq.layers
from keras.models import load_model
model = load_model(keras_model_path)


/home/ncgadmin/miniconda3/envs/devenv-hgq/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 31 variables whereas the saved optimizer has 60 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [26]:
y_keras = model.predict(x_test)

5188/5188 ━━━━━━━━━━━━━━━━━━━━ 2s 463us/step


In [27]:
# Save the model summary to a text file (Keras 3 style)
with open(os.path.join(model_dir, "summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda line: f.write(line + "\n"))

# Convert and synthesize with HLS4ML
Uses KV260 (xck26-sfvc784-2LV-c). You need to set the xpfm-path manually (it should be set based on env-path in source code?)

In [28]:
import hls4ml

hls_config = hls4ml.utils.config_from_keras_model(
    model, 
    backend='Vitis',
    )

hls_config['Model']['ReuseFactor'] = 1
hls_config['Model']['Strategy'] = 'latency'

hls_model = hls4ml.converters.convert_from_keras_model(
    model,    
    backend='Vitis',
    hls_config=hls_config,
    project_name=f'{model_to_test}_{model_revision}_hls4ml_prj_{hls4ml_revision}',
    output_dir=output_dir, 
    part='xck26-sfvc784-2LV-c',
)
hls_model.compile()
#hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True,to_file=os.path.join(output_dir, "model-plot.png"))

Check performance

In [29]:
y_hls = hls_model.predict(np.ascontiguousarray(x_test))

print("Difference in inference-calculations between Keras-model and HLS4ML-compiled model (first rows):")
for x,y in enumerate(y_keras[:5]):
    print(f"{y-y_hls[x]}")

#print(y_keras[:10])
#print(y_hls[:10])
print("Keras  Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
print("hls4ml Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))))



Difference in inference-calculations between Keras-model and HLS4ML-compiled model (first rows):
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
Keras  Accuracy: 0.755421686746988
hls4ml Accuracy: 0.755421686746988


In [30]:
hls_model.build(
    csim=False,
    #synth=True, 
    #bitfile=True
    ) 


****** vitis-run v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Wed Mar 18 19:22:26 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257
Sourcing Tcl script '/home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/1/hls4ml_prj_Vitis_latency_reusefactor1/build_prj.tcl'
INFO: [HLS 200-1510] Running: open_project hgq2_1_hls4ml_prj_Vitis_latency_reusefactor1_prj 
Resolution: For help on HLS 200-2182 see docs.amd.com/access/sources/dita/topic?Doc_Version=2025.2%20English&url=ug1448-hls-guidance&resourceid=200-2182.html
INFO: [HLS 200-10] Creating and opening solution '/home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/1/hls4ml_prj_Vitis_latency_reusefactor1/hgq2_1_hls4ml_prj_Vitis_latency_reusefactor1_prj'.
INFO: [HLS 200-1510] Running: set_top hgq2_1_hls4ml_prj_Vitis_latency_reusefactor1 
INFO: [HLS 20

{'CSynthesisReport': {'TargetClockPeriod': '5.00',
  'EstimatedClockPeriod': '3.573',
  'BestLatency': '9',
  'WorstLatency': '9',
  'IntervalMin': '1',
  'IntervalMax': '1',
  'DSP': '5',
  'FF': '2258',
  'LUT': '13096',
  'BRAM_18K': '0',
  'URAM': '0',
  'AvailableBRAM_18K': '288',
  'AvailableDSP': '1248',
  'AvailableFF': '234240',
  'AvailableLUT': '117120',
  'AvailableURAM': '64'}}

In [31]:
hls4ml.report.read_vivado_report(os.path.join(output_dir))

Found 1 solution(s) in /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/1/hls4ml_prj_Vitis_latency_reusefactor1/hgq2_1_hls4ml_prj_Vitis_latency_reusefactor1_prj.
Reports for solution "solution1":

C simulation report not found.
SYNTHESIS REPORT:
== Vitis HLS Report for 'hgq2_1_hls4ml_prj_Vitis_latency_reusefactor1'
* Date:           Wed Mar 18 19:23:13 2026

* Version:        2025.2 (Build 6295257 on Nov 14 2025)
* Project:        hgq2_1_hls4ml_prj_Vitis_latency_reusefactor1_prj
* Solution:       solution1 (Vivado IP Flow Target)
* Product family: zynquplus
* Target device:  xck26-sfvc784-2LV-c


== Performance Estimates
+ Timing: 
    * Summary: 
    +--------+---------+----------+------------+
    |  Clock |  Target | Estimated| Uncertainty|
    +--------+---------+----------+------------+
    |ap_clk  |  5.00 ns|  3.573 ns|     1.35 ns|
    +--------+---------+----------+------------+

+ Latency: 
    * Summary: 
    +---------+---------+-----------+-----------+-----+